# Postprocessing for the linear-elastic plate with a hole

This notebook runs the postprocessing script live. It is regenerated by
the `merge-docs-to-notebooks` GitHub Actions workflow from the benchmark
documentation in `docs/plate-with-hole.md`.


## 1. Imports

In [ ]:
import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 2. Locate benchmark result files

Results for each simulation tool live under the tool-specific results directory
(e.g. `fenics/results/`, `kratos/results/`, `extendablefem/results/`). Each
configuration produces a `solution_metrics.json` file that contains the
output metrics described in the documentation.

In [ ]:
REPO_ROOT = Path.cwd()
TOOL_DIRS = [REPO_ROOT / "fenics", REPO_ROOT / "kratos", REPO_ROOT / "extendablefem"]
METRIC_FILES = []
for tool_dir in TOOL_DIRS:
    if not tool_dir.exists():
        continue
    for config_dir in sorted((tool_dir / "results").glob("*")):
        if not config_dir.is_dir():
            continue
        for metrics_path in config_dir.rglob("solution_metrics.json"):
            METRIC_FILES.append((tool_dir.name, config_dir.name, metrics_path))
print(f"Found {len(METRIC_FILES)} metric files across all tools.")


## 3. Load and combine the metrics

In [ ]:
rows = []
for tool, config, path in METRIC_FILES:
    with open(path) as f:
        metrics = json.load(f)
    rows.append({
        "tool": tool,
        "config": config,
        **{k: metrics.get(k) for k in metrics},
    })
df = pd.DataFrame(rows)
df = df.sort_values(["tool", "config"]).reset_index(drop=True)
df


## 4. Convergence analysis

We expect the maximum von Mises stress to converge as the element size `h`
decreases. The relevant metric in `solution_metrics.json` is
`max_von_mises_stress[Pa]`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
if "max_von_mises_stress[Pa]" in df.columns:
    for tool, sub in df.groupby("tool"):
        sub = sub.sort_values("config")
        ax.plot(sub["config"], sub["max_von_mises_stress[Pa]"],
                marker="o", label=tool)
ax.set_xlabel("Configuration (mesh refinement)")
ax.set_ylabel("Max von Mises stress [Pa]")
ax.set_yscale("log")
ax.set_title("Convergence: max von Mises stress vs. mesh refinement")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
fig.tight_layout()
plt.show()


## 5. Displacement at the top-right corner

The `displacement_top_right_corner[m]` metric is a 2-vector `[ux, uy]`. We
plot both components as a function of the configuration to confirm that the
solution converges to the analytical Kirsch solution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
disp_col = "displacement_top_right_corner[m]"
if disp_col in df.columns:
    for tool, sub in df.groupby("tool"):
        sub = sub.sort_values("config")
        ux = [d[0] if d is not None else float("nan") for d in sub[disp_col]]
        uy = [d[1] if d is not None else float("nan") for d in sub[disp_col]]
        ax.plot(sub["config"], ux, marker="o", label=f"{tool} ux")
        ax.plot(sub["config"], uy, marker="s", linestyle="--", label=f"{tool} uy")
ax.set_xlabel("Configuration")
ax.set_ylabel("Displacement [m]")
ax.set_title("Top-right corner displacement convergence")
ax.legend()
ax.grid(True, ls="--", alpha=0.3)
fig.tight_layout()
plt.show()
